## 02b: Score Integrado ME BR (MI x ME) + Limite Final por Categoria

Este notebook executa **quatro passos sequenciais** sobre `apply_model_me_br`:

---

### Passo 1 — Score Integrado (MI + ME)

Consome `ds_catalog_dev.credit_engine.integrated_score_chile` (produzida pelo NB02b do pipeline Chile)
e atualiza colunas nos clientes **elegiveis em ambos os mercados no mesmo mes**.
Clientes apenas-ME nao sao tocados neste passo.

| Coluna atualizada | Valor | Fonte |
|-------------------|-------|-------|
| `integrated_score` | Score integrado (mi_share > 70% → MI, senao ME) | `integrated_score_chile` |
| `integrated_score_band` | Banda do score integrado | `integrated_score_chile` |
| `credit_limit` | `limit_mi_usd + limit_me_usd` (soma dos mercados) | `integrated_score_chile` |
| `credit_limit_clp` | `limit_mi_clp + limit_me_clp` (soma dos mercados) | `integrated_score_chile` |
| `adjusted_score_mi` | Score MI do cliente espelhado | `integrated_score_chile` |
| `limit_mi_usd` | Limite MI em USD | `integrated_score_chile` |
| `limit_mi_clp` | Limite MI em CLP | `integrated_score_chile` |
| `id_customer_mi` | cod_pessoa MI do cliente espelhado | `integrated_score_chile` |
| `market_scope` | `'MI_CHILE'` (indica cliente integrado MI+ME) | Fixo |

---

### Passo 1b — Propagacao de identidade MI+ME para todos os meses

**Problema:** clientes compartilhados MI+ME podem nao ser elegiveis simultaneamente em ambos os
modelos em determinados meses (ex: comprou so no MI em jan, so no ME em fev). O Passo 1 so atualiza
os meses onde ambos os modelos tem score — os demais ficam com `market_scope = NULL` e
`id_customer_mi = NULL`, fazendo o cliente parecer "apenas-ME" naquele periodo.

**Solucao (replica do NB02b Chile cell-8):** propaga `market_scope = 'MI_CHILE'` e `id_customer_mi`
para **todos os** `reference_month` do cliente ME que tem contraparte MI (via RUT), independente de
elegibilidade mensal. Fonte de verdade: `integrated_score_chile` (mapeamento estatico de identidade).
Toca apenas linhas onde esses campos ainda sao NULL (idempotente).

---

### Passo 1c — Propagacao do credit_limit combinado para membros do GE

**Problema:** o `credit_limit` em `apply_model_me_br` e um limite de **grupo economico** (GE) —
todos os membros do GE compartilham o mesmo teto. Quando um cliente MI+ME pertence a um GE com
outros membros apenas-ME, apos o Passo 1 apenas o cliente MI+ME tem o limite combinado (MI+ME);
os demais membros do GE continuam com o limite apenas-ME, gerando inconsistencia.

**Solucao:** para cada GE que possui pelo menos um cliente MI+ME atualizado no Passo 1
(identificado por `limit_mi_usd IS NOT NULL`), propaga o `credit_limit` e `credit_limit_clp`
combinados para todos os outros membros apenas-ME do mesmo GE e safra.

Executa apenas para meses onde o Passo 1 efetivamente atualizou o cliente MI+ME.
Os membros apenas-ME permanecem com `limit_mi_usd = NULL` e `market_scope = NULL` —
apenas o valor do limite do grupo e corrigido.

---

### Passo 2 — Teto por Categoria (credit_limit_end)

Aplica o teto de categoria sobre o `credit_limit` de **todos os clientes**
(independente de serem MI+ME ou apenas-ME). Executa depois dos Passos 1/1b/1c para
garantir que o `credit_limit` de todo o GE ja reflete a soma dos dois mercados.

**Alinhamento temporal:** join direto `apply_model.reference_month = customer_top_category.reference_quarter`.
Apesar do nome `reference_quarter`, a coluna e mensal com janela movel m+1,
identica ao espaco de `apply_model.reference_month`. Sem necessidade de DATE_TRUNC.

**Logica do teto por Grupo Economico:**

```
Para cada GE, avalia-se a categoria de todos os membros:
  teto_categoria_ge_usd = MAX(category_limit_usd) entre todos os membros do GE
  (o membro com a categoria de maior teto define o limite para todo o grupo)

credit_limit_end     = LEAST(credit_limit,     teto_categoria_ge_usd)
credit_limit_end_clp = LEAST(credit_limit_clp, teto_categoria_ge_usd * taxa_clp_usd)

Se o cliente nao tem categoria registrada: credit_limit_end = credit_limit (sem cap)
```

**Colunas atualizadas no Passo 2:**

| Coluna | Descricao |
|--------|-----------|
| `credit_limit_end` | Limite final USD: `LEAST(credit_limit, teto_categoria_ge_usd)` |
| `credit_limit_end_clp` | Limite final CLP: `LEAST(credit_limit_clp, teto_categoria_ge_usd * taxa)` |

---

### Ordem de execucao obrigatoria

```
Pipeline Chile:  NB01 → NB02 (apply_model_mi_chile) → NB02b (integrated_score_chile)
Pipeline ME BR:  NB01 → NB01b (category_limit) → NB02 (apply_model_me_br) → NB02b (este)

Dentro deste notebook:
  Passo 1  → score integrado (meses com ambos os mercados elegiveis)
  Passo 1b → identidade MI+ME para todos os meses (market_scope + id_customer_mi)
  Passo 1c → credit_limit combinado para demais membros do GE
  Passo 2  → teto por categoria sobre todos os clientes
```

### Dependencias

- `ds_catalog_dev.credit_engine.integrated_score_chile` (NB02b Chile) — scores, limites e crosswalk MI+ME
- `ds_catalog_dev.credit_engine.apply_model_me_br` (NB02 ME BR) — tabela destino de todos os MERGEs
- `ds_catalog_dev.credit_engine.customer_top_category_me_br` (NB01b) — categoria principal por cliente/trimestre
- `ds_catalog_dev.credit_engine.category_limit_me_br` (NB01b) — teto de limite por categoria/trimestre
- `de_data_lake_prd.financeiro.dw_tab_parametro_cotacao_cambial` — taxa CLP/USD para conversao do teto

In [ ]:
from datetime import date

dbutils.widgets.text('data_referencia', '', 'Data de Referencia')
data_referencia = dbutils.widgets.get('data_referencia')

effective_date = data_referencia if data_referencia else str(date.today())
print(f'Data de referencia: {effective_date}')

spark.sql(f"CREATE OR REPLACE TEMP VIEW config_pipeline AS SELECT CAST('{effective_date}' AS DATE) AS data_referencia")

Data de referencia: 2026-06-02


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.integrated_score_chile)                     AS isc_total_linhas,
  (SELECT CAST(MAX(reference_month) AS STRING)
   FROM ds_catalog_dev.credit_engine.integrated_score_chile)                                     AS isc_safra_max,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br)                          AS me_total_linhas,
  (SELECT CAST(MAX(reference_month) AS STRING)
   FROM ds_catalog_dev.credit_engine.apply_model_me_br)                                          AS me_safra_max,
  -- Clientes em comum esperados no Passo 1
  (SELECT COUNT(DISTINCT isc.id_customer_me)
   FROM ds_catalog_dev.credit_engine.integrated_score_chile isc
   INNER JOIN ds_catalog_dev.credit_engine.apply_model_me_br me
     ON isc.id_customer_me = me.id_customer
    AND isc.reference_month = me.reference_month)                         AS clientes_passo1_atualizar,
  -- Disponibilidade dos dados de categoria
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.customer_top_category_me_br)                AS ctc_total_linhas,
  (SELECT CAST(MAX(reference_quarter) AS STRING)
   FROM ds_catalog_dev.credit_engine.customer_top_category_me_br)                                AS ctc_trimestre_max,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.category_limit_me_br)                       AS cl_total_linhas,
  (SELECT CAST(MAX(reference_quarter) AS STRING)
   FROM ds_catalog_dev.credit_engine.category_limit_me_br)                                       AS cl_trimestre_max

In [ ]:
%sql
-- ====================================================================
-- MERGE PASSO 1: atualizar apply_model_me_br com scores integrados
-- ====================================================================
-- Fonte: integrated_score_chile (calculada pelo NB02b do pipeline Chile)
-- Apenas clientes com presenca em ambos os mercados sao atualizados.
-- Clientes apenas-ME nao sao tocados: mantem integrated_score = adjusted_score.
-- Idempotente: re-execucoes sobrescrevem com os mesmos valores.
-- ====================================================================
MERGE INTO ds_catalog_dev.credit_engine.apply_model_me_br AS target
USING (
  SELECT
    id_customer_mi,
    id_customer_me,
    reference_month,
    integrated_score,
    integrated_score_band,
    ROUND(credit_limit, 4) AS credit_limit_combined,
    ROUND(credit_limit_clp, 4) AS credit_limit_clp_combined,
    adjusted_score_mi,
    limit_mi_usd,
    limit_mi_clp
  FROM ds_catalog_dev.credit_engine.integrated_score_chile
) AS source
ON target.id_customer     = source.id_customer_me
   AND target.reference_month = source.reference_month
WHEN MATCHED THEN UPDATE SET
  target.integrated_score      = source.integrated_score,
  target.integrated_score_band = source.integrated_score_band,
  target.credit_limit          = source.credit_limit_combined,
  target.credit_limit_clp      = source.credit_limit_clp_combined,
  target.adjusted_score_mi     = source.adjusted_score_mi,
  target.limit_mi_usd          = source.limit_mi_usd,
  target.limit_mi_clp          = source.limit_mi_clp,
  target.id_customer_mi        = source.id_customer_mi,
  target.market_scope          = 'MI_CHILE',
  target.updated_at            = current_timestamp()

In [ ]:
%sql
-- ====================================================================
-- MERGE PASSO 1b: propagar id_customer_mi e market_scope para TODOS os meses
-- ====================================================================
-- Replica a logica do NB02b Chile (cell-8 de 02b_integrated_score_chile):
-- preenche id_customer_mi e market_scope = 'MI_CHILE' para TODOS os
-- reference_month do cliente ME que tem contraparte MI (via RUT),
-- INDEPENDENTE de elegibilidade mensal no modelo MI.
--
-- Problema resolvido: clientes compartilhados MI+ME que em determinados
-- meses so tem atividade em um dos mercados ficam sem market_scope nesses
-- meses apos o Passo 1 (pois integrated_score_chile so tem registros para
-- meses elegiveis nos dois modelos simultaneamente). Sem este passo esses
-- meses aparecem como "apenas-ME" no apply_model_me_br.
--
-- Fonte de verdade: integrated_score_chile (mapeamento estatico MI<->ME via RUT).
-- Toca apenas linhas onde market_scope IS NULL ou id_customer_mi IS NULL
-- para evitar sobrescrever meses ja corretamente preenchidos pelo Passo 1.
-- Idempotente: re-execucoes nao alteram linhas ja corretas.
-- ====================================================================
MERGE INTO ds_catalog_dev.credit_engine.apply_model_me_br AS target
USING (
  SELECT DISTINCT
    id_customer_me,
    id_customer_mi
  FROM ds_catalog_dev.credit_engine.integrated_score_chile
) AS source
ON target.id_customer = source.id_customer_me
WHEN MATCHED AND (target.market_scope IS NULL OR target.id_customer_mi IS NULL) THEN UPDATE SET
  target.market_scope   = 'MI_CHILE',
  target.id_customer_mi = source.id_customer_mi,
  target.updated_at     = current_timestamp()

In [ ]:
%sql
-- ====================================================================
-- MERGE PASSO 1c: propagar credit_limit combinado para demais membros do GE
-- ====================================================================
-- O credit_limit em apply_model_me_br e um limite de GRUPO ECONOMICO (GE):
-- todos os membros do GE devem compartilhar o mesmo teto de credito.
--
-- Problema: apos o Passo 1, apenas o cliente MI+ME tem credit_limit atualizado
-- para a soma MI+ME. Os outros membros apenas-ME do mesmo GE continuam com o
-- limite exclusivamente-ME calculado pelo NB02 — gerando inconsistencia no grupo.
--
-- Solucao: para cada GE que possui ao menos um cliente MI+ME atualizado no Passo 1
-- (identificado por limit_mi_usd IS NOT NULL), agrega o credit_limit combinado e
-- propaga para todos os membros apenas-ME do mesmo GE e safra.
--
-- Detalhes de design:
--   - Agrega via MAX no improvavel caso de >1 cliente MI+ME no mesmo GE
--     (MAX e conservador: pega o maior limite combinado disponivel)
--   - Membros apenas-ME permanecem com limit_mi_usd=NULL e market_scope=NULL:
--     apenas o valor de credit_limit e corrigido para consistencia do GE
--   - Executa apenas para meses onde o Passo 1 atualizou (limit_mi_usd IS NOT NULL):
--     meses sem elegibilidade MI+ME permanecem com o limite apenas-ME
--   - O Passo 2 (credit_limit_end) sera aplicado sobre o credit_limit ja corrigido
-- ====================================================================
MERGE INTO ds_catalog_dev.credit_engine.apply_model_me_br AS target
USING (
  WITH ge_com_mimeq AS (
    -- GEs que possuem ao menos um cliente MI+ME atualizado no Passo 1
    -- Agrega o credit_limit combinado por GE/safra
    SELECT
      id_customer_group_economic,
      reference_month,
      MAX(credit_limit)     AS credit_limit_combined,
      MAX(credit_limit_clp) AS credit_limit_clp_combined
    FROM ds_catalog_dev.credit_engine.apply_model_me_br
    WHERE limit_mi_usd IS NOT NULL  -- cliente MI+ME atualizado no Passo 1
    GROUP BY id_customer_group_economic, reference_month
  )
  -- Seleciona apenas os membros apenas-ME do GE (limit_mi_usd IS NULL)
  -- para receber o credit_limit combinado do grupo
  SELECT
    am.id_customer,
    am.reference_month,
    gc.credit_limit_combined,
    gc.credit_limit_clp_combined
  FROM ds_catalog_dev.credit_engine.apply_model_me_br am
  INNER JOIN ge_com_mimeq gc
    ON  am.id_customer_group_economic = gc.id_customer_group_economic
    AND am.reference_month            = gc.reference_month
  WHERE am.limit_mi_usd IS NULL  -- apenas membros apenas-ME do GE
) AS source
ON  target.id_customer     = source.id_customer
AND target.reference_month = source.reference_month
WHEN MATCHED THEN UPDATE SET
  target.credit_limit     = source.credit_limit_combined,
  target.credit_limit_clp = source.credit_limit_clp_combined,
  target.updated_at       = current_timestamp()

In [ ]:
%sql
-- ====================================================================
-- TAXA CLP/USD DO DIA
-- ====================================================================
-- Necessaria para converter teto_categoria_ge_usd em CLP ao calcular
-- credit_limit_end_clp = LEAST(credit_limit_clp, teto_usd * taxa_clp_usd)
-- Mesma logica de forward-fill do NB02 (de_data_lake_prd.financeiro.dw_tab_parametro_cotacao_cambial)
-- ====================================================================
CREATE OR REPLACE TEMP VIEW view_taxa_clp_hoje AS
WITH raw AS (
  SELECT
    CAST(valido_desde AS DATE) AS valido_desde,
    CAST(valido_ate   AS DATE) AS valido_ate,
    TRY_CAST(REPLACE(REPLACE(taxa_cambio, '/', ''), ',', '.') AS FLOAT) AS taxa_cambio
  FROM de_data_lake_prd.financeiro.dw_tab_parametro_cotacao_cambial
  WHERE moeda_procedencia = 'CLP'
    AND moeda_destino     = 'USD'
    AND taxa_cambio IS NOT NULL
    AND CAST(valido_desde AS DATE) <= CAST(valido_ate AS DATE)
),
exploded AS (
  SELECT CAST(exploded_date AS DATE) AS data_taxa, taxa_cambio
  FROM raw
  LATERAL VIEW EXPLODE(SEQUENCE(valido_desde, valido_ate, INTERVAL 1 DAY)) t AS exploded_date
),
date_spine AS (
  SELECT CAST(d AS DATE) AS data_taxa
  FROM (
    SELECT EXPLODE(SEQUENCE(
      (SELECT MIN(valido_desde) FROM raw),
      current_date(),
      INTERVAL 1 DAY
    )) AS d
  )
),
joined AS (
  SELECT ds.data_taxa, e.taxa_cambio,
    COUNT(e.taxa_cambio) OVER (
      ORDER BY ds.data_taxa ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS grp
  FROM date_spine ds
  LEFT JOIN exploded e ON e.data_taxa = ds.data_taxa
),
filled AS (
  SELECT data_taxa,
    FIRST_VALUE(taxa_cambio) OVER (PARTITION BY grp ORDER BY data_taxa) AS taxa_cambio
  FROM joined
)
SELECT taxa_cambio AS taxa_clp_usd
FROM filled
WHERE data_taxa = (SELECT MAX(data_taxa) FROM filled WHERE taxa_cambio IS NOT NULL)
LIMIT 1

In [ ]:
%sql
-- ====================================================================
-- TETO POR CATEGORIA — LOGICA DO GRUPO ECONOMICO
-- ====================================================================
--
-- Alinhamento temporal:
--   apply_model.reference_month e customer_top_category.reference_quarter
--   estao no mesmo espaco mensal (janela movel m+1). O nome "reference_quarter"
--   e apenas o nome da coluna na tabela de origem — o join e direto por igualdade,
--   sem DATE_TRUNC nem conversao de granularidade.
--
-- Logica do GE:
--   Cada membro do GE tem sua top_category e, consequentemente, seu category_limit_usd.
--   O teto do GE = MAX(category_limit_usd) entre todos os membros para o mesmo mes.
--   Ou seja: o membro com a categoria de maior limite "puxa" o teto para todo o grupo.
--
-- Se o cliente nao tem categoria registrada: teto = NULL → credit_limit_end = credit_limit
-- ====================================================================
CREATE OR REPLACE TEMP VIEW category_teto_ge AS
WITH

-- Categoria + limite por cliente/trimestre
-- customer_code em customer_top_category_me_br = id_customer (cod_pessoa)
customer_category AS (
  SELECT
    ctc.customer_code                     AS id_customer,
    ctc.id_customer_group_economic,
    ctc.reference_quarter,
    ctc.top_category,
    cl.category_limit_usd
  FROM ds_catalog_dev.credit_engine.customer_top_category_me_br ctc
  LEFT JOIN ds_catalog_dev.credit_engine.category_limit_me_br cl
    ON  ctc.top_category      = cl.category
    AND ctc.reference_quarter = cl.reference_quarter
),

-- Teto do GE: MAX(category_limit_usd) entre todos os membros do mesmo GE/trimestre
-- "avalia-se qual membro tem maior valor de categoria — esse define o limite do grupo"
ge_teto AS (
  SELECT
    id_customer_group_economic,
    reference_quarter,
    MAX(category_limit_usd) AS teto_categoria_ge_usd
  FROM customer_category
  GROUP BY id_customer_group_economic, reference_quarter
)

-- Junta o teto do GE de volta para cada cliente individual
-- (cada membro recebe o teto calculado para o seu grupo)
SELECT
  cc.id_customer,
  cc.reference_quarter,
  gt.teto_categoria_ge_usd
FROM customer_category cc
INNER JOIN ge_teto gt
  ON  cc.id_customer_group_economic = gt.id_customer_group_economic
  AND cc.reference_quarter          = gt.reference_quarter

In [ ]:
%sql
-- ====================================================================
-- MERGE PASSO 2: credit_limit_end para TODOS os clientes
-- ====================================================================
-- Executa sobre toda a apply_model_me_br (nao apenas clientes MI+ME).
-- O credit_limit ja reflete a soma MI+ME para clientes compartilhados
-- (Passo 1 ja rodou), entao o LEAST e aplicado sobre o valor correto.
--
-- Regra:
--   credit_limit_end     = LEAST(credit_limit,     teto_categoria_ge_usd)
--   credit_limit_end_clp = LEAST(credit_limit_clp, teto_categoria_ge_usd * taxa_clp_usd)
--
-- Se cliente nao tem categoria (LEFT JOIN retorna NULL):
--   COALESCE(teto, credit_limit) = credit_limit → sem cap → credit_limit_end = credit_limit
--
-- Idempotente: re-execucoes recalculam com os mesmos valores.
-- ====================================================================
MERGE INTO ds_catalog_dev.credit_engine.apply_model_me_br AS target
USING (
  SELECT
    am.id_customer,
    am.reference_month,
    -- USD: menor entre o limite calculado e o teto da categoria do GE
    ROUND(
      LEAST(
        am.credit_limit,
        COALESCE(ct.teto_categoria_ge_usd, am.credit_limit)
      ),
    4) AS credit_limit_end,
    -- CLP: converte o teto USD para CLP e aplica o mesmo LEAST
    ROUND(
      LEAST(
        am.credit_limit_clp,
        COALESCE(ct.teto_categoria_ge_usd * taxa.taxa_clp_usd, am.credit_limit_clp)
      ),
    4) AS credit_limit_end_clp
  FROM ds_catalog_dev.credit_engine.apply_model_me_br am
  -- Join direto: reference_month = reference_quarter (mesmo espaco mensal m+1)
  LEFT JOIN category_teto_ge ct
    ON  am.id_customer        = ct.id_customer
    AND am.reference_month    = ct.reference_quarter
  CROSS JOIN view_taxa_clp_hoje taxa
) AS source
ON target.id_customer     = source.id_customer
   AND target.reference_month = source.reference_month
WHEN MATCHED THEN UPDATE SET
  target.credit_limit_end       = source.credit_limit_end,
  target.credit_limit_end_clp   = source.credit_limit_end_clp,
  target.updated_at             = current_timestamp()

In [ ]:
%sql
-- ====================================================================
-- SANITY CHECKS: apply_model_me_br apos NB02b (Passos 1, 1b, 1c e 2)
-- ====================================================================
SELECT
  -- Volume geral
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br)                          AS total_linhas,
  (SELECT COUNT(DISTINCT reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br)   AS total_safras,
  (SELECT CAST(MAX(reference_month) AS STRING)
   FROM ds_catalog_dev.credit_engine.apply_model_me_br)                                          AS safra_max,

  -- PASSO 1: score integrado — meses com ambos os mercados elegiveis
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE limit_mi_usd IS NOT NULL)                                        AS linhas_mi_me_passo1,
  (SELECT COUNT(DISTINCT id_customer) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE limit_mi_usd IS NOT NULL)                                        AS clientes_distintos_mi_me,

  -- PASSO 1: qualidade do integrated_score
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE integrated_score IS NULL)                                        AS nulls_integrated_score,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE integrated_score < 0 OR integrated_score > 1)                   AS integrated_score_out_of_range,

  -- PASSO 1b: identidade MI+ME propagada para todos os meses
  -- Esperado: clientes_com_id_customer_mi > linhas_mi_me_passo1
  -- (ha meses com market_scope mas sem limit_mi — meses parcialmente ativos)
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE market_scope = 'MI_CHILE')                                       AS linhas_market_scope_mi_chile,
  (SELECT COUNT(DISTINCT id_customer) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE market_scope = 'MI_CHILE')                                       AS clientes_distintos_market_scope,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE id_customer_mi IS NOT NULL)                                      AS linhas_com_id_customer_mi,
  -- Meses com market_scope mas sem limit_mi (clientes elegiveis so em 1 mercado naquele mes)
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE market_scope = 'MI_CHILE' AND limit_mi_usd IS NULL)              AS meses_market_scope_sem_limit_mi,
  -- Consistencia: market_scope e id_customer_mi devem ser preenchidos juntos
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE (market_scope IS NOT NULL AND id_customer_mi IS NULL)
      OR (market_scope IS NULL     AND id_customer_mi IS NOT NULL))       AS inconsistencias_market_scope,

  -- PASSO 1c: propagacao para membros do GE
  -- GEs distintos que tem ao menos 1 cliente MI+ME
  (SELECT COUNT(DISTINCT id_customer_group_economic)
   FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE limit_mi_usd IS NOT NULL)                                        AS ges_com_cliente_mimeq,
  -- Membros apenas-ME de GEs com cliente MI+ME (afetados pelo Passo 1c)
  -- Reescrito como INNER JOIN para evitar IllegalArgumentException de self-join correlacionado no Spark
  (SELECT COUNT(*)
   FROM ds_catalog_dev.credit_engine.apply_model_me_br am
   INNER JOIN (
     SELECT DISTINCT id_customer_group_economic, reference_month
     FROM ds_catalog_dev.credit_engine.apply_model_me_br
     WHERE limit_mi_usd IS NOT NULL
   ) mimeq
     ON  am.id_customer_group_economic = mimeq.id_customer_group_economic
     AND am.reference_month            = mimeq.reference_month
   WHERE am.limit_mi_usd IS NULL)                                         AS linhas_ge_apenas_me_atualizadas,

  -- PASSO 2: cobertura de credit_limit_end
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE credit_limit_end IS NOT NULL)                                    AS clientes_com_credit_limit_end,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE credit_limit_end IS NULL)                                        AS clientes_sem_credit_limit_end,

  -- PASSO 2: quantos tiveram o limite capado pela categoria na ultima safra
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE credit_limit_end < credit_limit
     AND reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br))
                                                                          AS clientes_capados_ultima_safra,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE credit_limit_end = credit_limit
     AND reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br))
                                                                          AS clientes_abaixo_teto_ultima_safra,

  -- Medias de limite para validacao de magnitude
  (SELECT ROUND(AVG(credit_limit), 2) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br))
                                                                          AS avg_credit_limit,
  (SELECT ROUND(AVG(credit_limit_end), 2) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE credit_limit_end IS NOT NULL
     AND reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br))
                                                                          AS avg_credit_limit_end,

  -- Alinhamento com integrated_score_chile
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.integrated_score_chile
   WHERE reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.integrated_score_chile))
                                                                          AS isc_linhas_ultima_safra,
  (SELECT COUNT(*) FROM ds_catalog_dev.credit_engine.apply_model_me_br
   WHERE limit_mi_usd IS NOT NULL
     AND reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br))
                                                                          AS me_linhas_mi_me_ultima_safra

In [ ]:
%sql
-- Amostra para validacao manual: clientes onde o teto de categoria
-- efetivamente reduziu o limite (credit_limit_end < credit_limit)
SELECT
  id_customer,
  customer_name,
  reference_month,
  score_band,
  ROUND(credit_limit, 2)                    AS credit_limit_antes,
  ROUND(credit_limit_end, 2)                AS credit_limit_end,
  ROUND(credit_limit - credit_limit_end, 2) AS reducao_usd,
  CASE WHEN limit_mi_usd IS NOT NULL THEN 'MI+ME' ELSE 'Apenas-ME' END AS tipo_cliente,
  market_scope,
  id_customer_mi
FROM ds_catalog_dev.credit_engine.apply_model_me_br
WHERE credit_limit_end < credit_limit
  AND reference_month = (SELECT MAX(reference_month) FROM ds_catalog_dev.credit_engine.apply_model_me_br)
ORDER BY reducao_usd DESC
LIMIT 50

In [ ]:
%sql
select * from ds_catalog_dev.credit_engine.apply_model_me_br where reference_month = '2026-05-01' and id_customer in('403702','403800')

--403702  403800

In [ ]:
%sql
SELECT *
FROM ds_catalog_dev.credit_engine.integrated_score_chile
WHERE id_customer_mi = 1104616
  AND reference_month = '2026-05-01'

In [ ]:
%sql
WITH ge_com_mi AS (
    -- grupos econômicos que têm pelo menos 1 membro integrado MI+ME
    SELECT DISTINCT id_customer_group_economic
    FROM ds_catalog_dev.credit_engine.apply_model_me_br
    WHERE reference_month = '2026-05-01'
      AND id_customer_mi IS NOT NULL
),
ge_multi_membro AS (
    -- grupos econômicos com mais de 1 cliente ME nesta safra
    SELECT id_customer_group_economic
    FROM ds_catalog_dev.credit_engine.apply_model_me_br
    WHERE reference_month = '2026-05-01'
    GROUP BY id_customer_group_economic
    HAVING COUNT(*) > 1
)
SELECT
    me.id_customer_group_economic,
    me.id_customer,
    me.id_customer_mi,
    me.market_scope,
    me.reference_month,
    -- scores
    me.adjusted_score,
    me.score_band,
    me.adjusted_score_mi,
    me.integrated_score,
    me.integrated_score_band,
    -- limites antes do teto
    me.credit_limit          AS limit_ge_usd,
    me.credit_limit_clp      AS limit_ge_clp,
    me.limit_mi_usd,
    me.limit_mi_clp,
    -- limites finais (após teto categoria)
    me.credit_limit_end      AS limit_final_usd,
    me.credit_limit_end_clp  AS limit_final_clp
FROM ds_catalog_dev.credit_engine.apply_model_me_br me
INNER JOIN ge_com_mi     g  ON me.id_customer_group_economic = g.id_customer_group_economic
INNER JOIN ge_multi_membro gm ON me.id_customer_group_economic = gm.id_customer_group_economic
WHERE me.reference_month = '2026-05-01'
ORDER BY me.id_customer_group_economic, me.id_customer

---

## Simulacao — Penalizacao por Inadimplencia sobre parcela ME

Aplica um fator de suavizacao `fator_def` **somente sobre a parcela ME** do `credit_limit_end`.
A parcela MI (`limit_mi_usd`) entra somada depois, sem penalizacao.

**Formula:**
```
taxa         = months_defaulted / months_with_billing
fator_def    = GREATEST(0.60, POWER(1 - taxa, 1/N))   -- N configuravel em params
limit_me     = credit_limit_end - limit_mi_usd         -- parcela ME isolada
limit_total  = LEAST( limit_me x fator_def + limit_mi_usd , credit_limit_end )
```

| taxa_inadimp | N=5   |
|--------------|-------|
| 0%           | 1.000 |
| 30%          | 0.931 |
| 50%          | 0.871 |
| 80%          | 0.725 |
| 100% (floor) | 0.600 |

> **Para alterar N:** mudar o valor `5` no CTE `params` abaixo.

In [ ]:
%sql
-- ============================================================
-- SIMULACAO — Penalizacao por Inadimplencia (parcela ME)
-- ============================================================
-- fator_def = GREATEST(0.60, POWER(1 - taxa_inadimplencia, 1/N))
--   0% atraso  → 1.00 (sem penalizacao)
--   100% atraso → 0.60 (floor fixo)
--
-- Multiplicador incide APENAS sobre a parcela ME:
--   limit_me     = credit_limit_end - limit_mi_usd
--   limit_total  = LEAST(limit_me * fator_def + limit_mi_usd, credit_limit_end)
--
-- Para alterar o fator de suavizacao: editar N no CTE params abaixo.
-- ============================================================
WITH

-- ── PARAMETRO: fator de suavizacao N ──────────────────────────────────────────
params AS (
  SELECT 5 AS n_suavizacao            -- <<< ALTERE AQUI (ex: 5 = mais rigido, 9 = mais suave)
),

-- ── SAFRA MAIS RECENTE ────────────────────────────────────────────────────────
ultima_safra AS (
  SELECT MAX(reference_month) AS safra
  FROM ds_catalog_dev.credit_engine.apply_model_me_br
),

-- ── BASE: motor + features abt (mes M-1) ─────────────────────────────────────
base AS (
  SELECT
    am.id_customer,
    am.customer_name,
    am.country                                                     AS pais,
    am.reference_month                                             AS safra,
    am.score_band,
    COALESCE(am.limit_mi_usd, 0)                                   AS limit_mi_usd,
    COALESCE(am.limit_mi_clp, 0)                                   AS limit_mi_clp,
    -- Parcela ME = limite final capado menos a parte MI
    am.credit_limit_end     - COALESCE(am.limit_mi_usd, 0)         AS limit_me_usd,
    am.credit_limit_end_clp - COALESCE(am.limit_mi_clp, 0)         AS limit_me_clp,
    am.credit_limit_end                                            AS credit_limit_end_atual,
    am.credit_limit_end_clp                                        AS credit_limit_end_clp_atual,
    inf.months_with_billing,
    inf.months_defaulted,
    -- Taxa de inadimplencia: defaulted / billing, travada em [0, 1]
    CASE
      WHEN COALESCE(inf.months_with_billing, 0) = 0 THEN 0.0
      ELSE LEAST(1.0, CAST(COALESCE(inf.months_defaulted, 0) AS DOUBLE)
                      / inf.months_with_billing)
    END                                                            AS taxa_inadimplencia
  FROM ds_catalog_dev.credit_engine.apply_model_me_br   am
  JOIN ds_catalog_dev.credit_engine.abt_inference_me_br inf
    ON  inf.id_customer     = am.id_customer
    AND inf.reference_month = ADD_MONTHS(am.reference_month, -1)
  WHERE am.reference_month = (SELECT safra FROM ultima_safra)
),

-- ── CALCULO DO FATOR E LIMITES PENALIZADOS ────────────────────────────────────
resultado AS (
  SELECT
    b.*,
    p.n_suavizacao,
    -- fator de suavizacao com floor fixo de 0.60
    GREATEST(0.6, POWER(1.0 - b.taxa_inadimplencia, 1.0 / p.n_suavizacao)) AS fator_def
  FROM base b
  CROSS JOIN params p
)

-- ── RESULTADO FINAL ───────────────────────────────────────────────────────────
SELECT
  id_customer,
  customer_name,
  pais,
  safra,
  score_band,
  CASE WHEN limit_mi_usd > 0 THEN 'MI+ME' ELSE 'Apenas-ME' END   AS tipo_cliente,
  months_with_billing,
  months_defaulted,
  ROUND(taxa_inadimplencia * 100, 1)                             AS taxa_inadimp_pct,
  n_suavizacao                                                   AS N,
  ROUND(fator_def, 4)                                            AS fator_def,

  -- Componentes do limite
  ROUND(limit_me_usd, 2)                                         AS limit_me_usd,
  ROUND(limit_mi_usd, 2)                                         AS limit_mi_usd,
  ROUND(limit_me_usd * fator_def, 2)                             AS limit_me_penalizado_usd,

  -- Limite final atual vs penalizado (cap no teto credit_limit_end)
  ROUND(credit_limit_end_atual, 2)                               AS limit_final_atual_usd,
  ROUND(
    LEAST(
      limit_me_usd * fator_def + limit_mi_usd,
      COALESCE(credit_limit_end_atual, limit_me_usd * fator_def + limit_mi_usd)
    ), 2)                                                        AS limit_final_penalizado_usd,
  ROUND(
    LEAST(
      limit_me_clp * fator_def + limit_mi_clp,
      COALESCE(credit_limit_end_clp_atual, limit_me_clp * fator_def + limit_mi_clp)
    ), 2)                                                        AS limit_final_penalizado_clp,

  -- Delta absoluto e percentual
  ROUND(
    LEAST(
      limit_me_usd * fator_def + limit_mi_usd,
      COALESCE(credit_limit_end_atual, limit_me_usd * fator_def + limit_mi_usd)
    ) - credit_limit_end_atual, 2)                               AS delta_usd,
  ROUND(
    (
      LEAST(
        limit_me_usd * fator_def + limit_mi_usd,
        COALESCE(credit_limit_end_atual, limit_me_usd * fator_def + limit_mi_usd)
      ) - credit_limit_end_atual
    ) / NULLIF(credit_limit_end_atual, 0) * 100, 1)              AS delta_pct

FROM resultado
ORDER BY limit_final_atual_usd DESC